# 03 — FastAPI App & Testing
## Fraud Detection MLOps Platform

This notebook:
1. Starts the FastAPI server as a background process
2. Tests all API endpoints
3. Runs batch predictions on the test set
4. Verifies the API matches the model's direct predictions

### The API file is at: `api/main.py`
### Once running, interactive docs are at: http://127.0.0.1:8000/docs

---
## Step 1 — Install & verify dependencies

In [1]:
import subprocess, sys

# Make sure fastapi and uvicorn are installed
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "fastapi", "uvicorn[standard]", "httpx", "-q"])
print("FastAPI dependencies ready")

FastAPI dependencies ready


---
## Step 2 — Start the API server in the background

Uvicorn is launched as a background subprocess so the notebook stays interactive.
The server starts at http://127.0.0.1:8000

In [2]:
import subprocess
import time
import sys
from pathlib import Path

# Change working directory to project root (one level up from notebooks/)
import os
os.chdir("..")
print("Working directory:", os.getcwd())

# Start uvicorn as a background process
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
print("Starting API server...")
time.sleep(4)

if server.poll() is None:
    print("   API server running at http://127.0.0.1:8000")
    print("   Interactive docs: http://127.0.0.1:8000/docs")
else:
    err = server.stderr.read().decode()
    print("   Server failed to start:")
    print(err)

Working directory: F:\physical science\4th year\Internship\Task 20\fraud-detection-mlops-platform
Starting API server...
   API server running at http://127.0.0.1:8000
   Interactive docs: http://127.0.0.1:8000/docs


---
## Step 3 — Test the health and root endpoints

In [3]:
import httpx

base_url = "http://127.0.0.1:8000"

# Test root endpoint
resp = httpx.get(f"{base_url}/")
print("GET /")
print(f"  Status: {resp.status_code}")
print(f"  Body:   {resp.json()}")
print()

# Test health endpoint
resp = httpx.get(f"{base_url}/health")
print("GET /health")
print(f"  Status: {resp.status_code}")
print(f"  Body:   {resp.json()}")
print()

# Test model-info endpoint
resp = httpx.get(f"{base_url}/model-info")
print("GET /model-info")
print(f"  Status: {resp.status_code}")
info = resp.json()
print(f"  Model:    {info['model_file']}")
print(f"  Features: {info['num_features']}")

GET /
  Status: 200
  Body:   {'message': 'Fraud Detection API is running', 'docs': '/docs', 'health': '/health'}

GET /health
  Status: 200
  Body:   {'status': 'healthy', 'model': 'best_model.pkl', 'features': 12, 'timestamp': '2026-05-29T05:53:14.858885'}

GET /model-info
  Status: 200
  Model:    best_model.pkl
  Features: 12


---
## Step 4 — Test the /predict endpoint

Send a sample transaction and check the response.
Both a normal transaction and a suspicious one are tested.

In [4]:
# Test 1 — Normal transaction (low fraud risk)
normal_transaction = {
    "step": 10,
    "amount": 500.0,
    "oldbalanceOrg": 5000.0,
    "newbalanceOrig": 4500.0,
    "oldbalanceDest": 1000.0,
    "newbalanceDest": 1500.0,
    "type_encoded": 0,
    "orig_balance_diff": 500.0,
    "dest_balance_diff": 500.0,
    "orig_balance_zero": 0,
    "amount_to_balance_ratio": 0.1,
    "hour_of_day": 10
}

resp = httpx.post(f"{base_url}/predict", json=normal_transaction)
print("POST /predict — Normal transaction")
print(f"  Status: {resp.status_code}")
result = resp.json()
print(f"  is_fraud:          {result['is_fraud']}")
print(f"  fraud_probability: {result['fraud_probability']}")
print(f"  risk_level:        {result['risk_level']}")
print()

POST /predict — Normal transaction
  Status: 200
  is_fraud:          False
  fraud_probability: 0.0
  risk_level:        LOW



In [5]:
# Test 2 — Suspicious transaction (high fraud risk)
# Hallmarks of fraud: sender balance goes to zero, large amount, TRANSFER type
suspicious_transaction = {
    "step": 1,
    "amount": 180000.0,
    "oldbalanceOrg": 180000.0,
    "newbalanceOrig": 0.0,              # balance wiped out
    "oldbalanceDest": 0.0,
    "newbalanceDest": 0.0,              # destination doesn't receive it
    "type_encoded": 1,                  # TRANSFER
    "orig_balance_diff": 180000.0,
    "dest_balance_diff": 0.0,
    "orig_balance_zero": 1,             # went to zero
    "amount_to_balance_ratio": 1.0,     # 100% of balance
    "hour_of_day": 1
}

resp = httpx.post(f"{base_url}/predict", json=suspicious_transaction)
print("POST /predict — Suspicious transaction")
print(f"  Status: {resp.status_code}")
result = resp.json()
print(f"  is_fraud:          {result['is_fraud']}")
print(f"  fraud_probability: {result['fraud_probability']}")
print(f"  risk_level:        {result['risk_level']}")

POST /predict — Suspicious transaction
  Status: 200
  is_fraud:          True
  fraud_probability: 0.9986
  risk_level:        HIGH


---
## Step 5 — Batch test against the real test set

Send all 40,000 test transactions through the API and compare
API predictions against the model's direct predictions.
This confirms the API is correctly wrapping the model.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the test data
X_test = pd.read_csv("data/processed/X_test.csv")
y_test = pd.read_csv("data/processed/y_test.csv").squeeze()

# Run batch predictions directly through the model (ground truth)
import joblib
model = joblib.load("models/best_model.pkl")
direct_preds = model.predict(X_test)
direct_proba = model.predict_proba(X_test)[:, 1]

print(f"Direct model predictions on {len(X_test):,} test rows:")
print(f"  Predicted fraud:     {direct_preds.sum():,}")
print(f"  Actual fraud:        {y_test.sum():,}")
print(f"  Avg fraud probability: {direct_proba.mean():.4f}")

Direct model predictions on 40,000 test rows:
  Predicted fraud:     173
  Actual fraud:        119
  Avg fraud probability: 0.0045


In [7]:
# Send a sample of 10 transactions through the API and verify they match
import json

sample_indices = X_test.sample(10, random_state=42).index
mismatches = 0

print("Comparing API predictions vs direct model predictions (10 samples):\n")
print(f"{'Idx':>6} | {'Direct Pred':>11} | {'API Pred':>8} | {'API Prob':>8} | {'Match':>5}")
print("-" * 55)

for idx in sample_indices:
    row = X_test.loc[idx].to_dict()

    # Round floats to avoid JSON serialization issues
    row = {k: round(float(v), 6) if isinstance(v, float) else int(v)
           for k, v in row.items()}

    resp = httpx.post(f"{base_url}/predict", json=row)
    api_result = resp.json()

    direct = int(direct_preds[idx])
    api    = int(api_result["is_fraud"])
    match  = "YES" if direct == api else "NO"

    if direct != api:
        mismatches += 1

    print(f"{idx:>6} | {direct:>11} | {api:>8} | {api_result['fraud_probability']:>8.4f} | {match}")

print()
print(f"Mismatches: {mismatches}/10")
print("API matches model perfectly!" if mismatches == 0 else "Check feature ordering in api/main.py")

Comparing API predictions vs direct model predictions (10 samples):

   Idx | Direct Pred | API Pred | API Prob | Match
-------------------------------------------------------
 32823 |           0 |        0 |   0.0000 | YES
 16298 |           0 |        0 |   0.0000 | YES
 28505 |           0 |        0 |   0.0000 | YES
  6689 |           0 |        0 |   0.0004 | YES
 26893 |           0 |        0 |   0.0000 | YES
 36572 |           0 |        0 |   0.0000 | YES
 12335 |           0 |        0 |   0.0000 | YES
 29591 |           0 |        0 |   0.0001 | YES
 18948 |           0 |        0 |   0.0000 | YES
 31067 |           0 |        0 |   0.0000 | YES

Mismatches: 0/10
API matches model perfectly!


---
## Step 6 — Stop the server

In [8]:
# Gracefully stop the background server
server.terminate()
server.wait()
print("API server stopped")
print()
print("=" * 50)
print("API TESTING COMPLETE")
print("=" * 50)
print("Endpoints verified:  /, /health, /model-info, /predict")
print()
print("To run the API manually anytime:")
print("cd to project root")
print("uvicorn api.main:app --reload")
print()
print("Next step -> open 04_drift_monitoring.ipynb")

API server stopped

API TESTING COMPLETE
Endpoints verified:  /, /health, /model-info, /predict

To run the API manually anytime:
cd to project root
uvicorn api.main:app --reload

Next step -> open 04_drift_monitoring.ipynb


---
---